# Dimensión de provincias

Tabla de referencia con las 24 jurisdicciones argentinas más el residual `OPERAC.RESIDENTES EN EL EXTERIOR` que usa el BCRA en `distribgeo`.

A diferencia de las otras dimensiones, esta se genera a mano (no sale de un archivo del IEF). Se define el crosswalk entre el nombre literal usado por el BCRA y el código ISO 3166-2:AR estándar.

Output: `data/interim/dimensiones/dim_provincias.parquet`.
Fuente primaria del listado: `data/external/crosswalks/provincias_iso.csv` (creado en este notebook si no existe).

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "utils"))
from paths import RAW, INTERIM, DIMENSIONES, PANELES, REPO

import pandas as pd
import duckdb

CROSSWALK = REPO / "data/external/crosswalks/provincias_iso.csv"
OUT = DIMENSIONES / "dim_provincias.parquet"
OUT.parent.mkdir(parents=True, exist_ok=True)
CROSSWALK.parent.mkdir(parents=True, exist_ok=True)

## Creación del crosswalk

Si no existe el CSV, se crea con los 24 códigos ISO y el nombre BCRA. Los nombres BCRA se toman del archivo `distribgeo` del dump 202601.

In [ ]:
provincias = [
    ("AR-C", "CIUDAD AUTÓNOMA DE BUENOS AIRES",     "CABA"),
    ("AR-B", "BUENOS AIRES",                        "Buenos Aires"),
    ("AR-K", "CATAMARCA",                           "Catamarca"),
    ("AR-H", "CHACO",                               "Chaco"),
    ("AR-U", "CHUBUT",                              "Chubut"),
    ("AR-X", "CORDOBA",                             "Córdoba"),
    ("AR-W", "CORRIENTES",                          "Corrientes"),
    ("AR-E", "ENTRE RIOS",                          "Entre Ríos"),
    ("AR-P", "FORMOSA",                             "Formosa"),
    ("AR-Y", "JUJUY",                               "Jujuy"),
    ("AR-L", "LA PAMPA",                            "La Pampa"),
    ("AR-F", "LA RIOJA",                            "La Rioja"),
    ("AR-M", "MENDOZA",                             "Mendoza"),
    ("AR-N", "MISIONES",                            "Misiones"),
    ("AR-Q", "NEUQUEN",                             "Neuquén"),
    ("AR-R", "RIO NEGRO",                           "Río Negro"),
    ("AR-A", "SALTA",                               "Salta"),
    ("AR-J", "SAN JUAN",                            "San Juan"),
    ("AR-D", "SAN LUIS",                            "San Luis"),
    ("AR-Z", "SANTA CRUZ",                          "Santa Cruz"),
    ("AR-S", "SANTA FE",                            "Santa Fe"),
    ("AR-G", "SANTIAGO DEL ESTERO",                 "Santiago del Estero"),
    ("AR-V", "TIERRA DEL FUEGO",                    "Tierra del Fuego"),
    ("AR-T", "TUCUMAN",                             "Tucumán"),
    ("EXT",  "OPERAC.RESIDENTES EN EL EXTERIOR",    "Exterior"),
]

dim = pd.DataFrame(provincias, columns=["iso_codigo", "nombre_bcra", "nombre_largo"])
dim["es_jurisdiccion_argentina"] = dim["iso_codigo"] != "EXT"
dim.to_csv(CROSSWALK, index=False)
dim.to_parquet(OUT, index=False)
print(f"Escrito: {OUT.relative_to(REPO)}")
print(f"Crosswalk: {CROSSWALK.relative_to(REPO)}")

## Validaciones

In [ ]:
assert len(dim) == 25, "Se esperan 24 provincias + exterior"
assert dim["iso_codigo"].is_unique
assert dim["nombre_bcra"].is_unique
assert dim["es_jurisdiccion_argentina"].sum() == 24
print("Validaciones OK")

## Muestra

In [ ]:
duckdb.sql(f"select * from '{OUT}' order by iso_codigo").df()